# Lesson 6 — Build a three-tier router (and then question the number)

**Module 3 · ~15 minutes · API key required · the intellectual centre of the course**

Routing is the largest cost lever and the least deployed one. Glean's CEO estimated ~95% of enterprise usage still hits the most expensive frontier model. We will reproduce the win — then show why the raw saving can be a lie once cleanup cost and success rate are in the picture.

Uses the three tiers your key can actually call (`MODEL_FLOOR` / `MODEL_MID` / `MODEL_FRONTIER`). If you only have one model, set all three env vars to it: the **cost-per-solved-task reframe still lands**, which is the payload.

> **Presenting:** after the cascade result, say "we just cut cost by X% — now let me show you why that number might be a lie" *before* you run the reframe.

### What you will be able to do

1. Run the same mixed workload all-on-mid (what most teams ship) vs a cheap-first cascade.
2. Read the routing mix: how many queries actually needed the expensive model.
3. Divide by success rate, add cleanup cost, and watch the ranking flip as cleanup goes from $0 to $50.


### How to work through this notebook

Run cells **top to bottom**. Each section tells you what is about to happen *before* you run the code.

| Marker | What it means |
|---|---|
| **About to happen** | What the next cell will do |
| **Watch for** | The number or field that makes the point — pause on it |
| **Why it matters** | The Monday-morning decision this should change |
| **Presenting:** | Live-demo cue. Students: treat this as the takeaway |

A **cost ledger** prints at the end of every notebook that spends money.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
print(f"\nLive provider: {cfg.provider}")
print("Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.")


  Provider : none (offline)
  Arithmetic cells still run. Live cells will use rehearsal fallbacks.
  Add OPENAI_API_KEY or ANTHROPIC_API_KEY to .env for live calls.
  Rate card: verified 5 Sep 2026 — re-check before presenting.

Live provider: offline
Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.


The cell above loads `.env`, chooses **OpenAI or Anthropic** from the key you have, and prints the three model tiers this notebook will call.

**Watch for:** a banner with `Provider`, `floor`, `mid`, `frontier`.
- If it names a vendor, live cells will spend a few cents.
- If it says `offline`, arithmetic still runs. Live cells print a rehearsal fallback instead of crashing — useful on a plane, not a substitute for a real key on caching / routing / eval lessons.

Switch vendor by setting `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and re-running that cell.


Confirm the three rungs your key will actually call. If any 404, edit `MODEL_FLOOR` / `MODEL_MID` / `MODEL_FRONTIER` in `.env` and re-run setup.


In [2]:
print("Routing ladder for this key:")
print(f"  floor        {MODELS.floor}")
print(f"  near-frontier {MODELS.mid}")
print(f"  frontier     {MODELS.frontier}")
print()
print("Edit MODEL_FLOOR / MODEL_MID / MODEL_FRONTIER in .env if any of these 404.")


Routing ladder for this key:
  floor        claude-haiku-4-5
  near-frontier claude-sonnet-5
  frontier     claude-opus-5

Edit MODEL_FLOOR / MODEL_MID / MODEL_FRONTIER in .env if any of these 404.


---
## 1. A mixed workload — 70% simple, 20% medium, 10% hard

**About to happen.** Twelve tasks: 8 extraction (floor work), 3 grounded policy decisions (medium), 1 multi-constraint synthesis (hard). Production telemetry often looks like this — it is a **planning prior** echoed in routing papers (">70% routine"), not a measured industry statistic you should put on a board slide without a source.

**Watch for:** the class counts. Routing only pays if most traffic is actually easy. If your traffic is 80% hard, a cascade will cost *more* because of the extra probe calls.

Twelve tasks keep the room moving. Bump the counts if you have time and budget.


In [3]:
TASKS = (
    [dict(q=f'Extract the order id from: "Order NW-{1000 + i} was delayed 3 days." '
            "Reply with the id only.",
          gold=f"NW-{1000 + i}", klass="simple") for i in range(8)]
    + [dict(q="Policy: refunds under $500 auto-approve if delay > 48h and fewer than 3 prior claims. "
            f"Customer: ${300 + i * 40} claim, 60h delay, {i % 4} prior claims. Auto-approve? "
            "Answer YES or NO first.",
            gold="YES" if (300 + i * 40) < 500 and (i % 4) < 3 else "NO",
            klass="medium") for i in range(3)]
    + [dict(q="Three shipments: A delayed 50h claim $480 with 2 prior claims; B delayed 20h claim "
            "$100 with 0 prior; C delayed 72h claim $900 with 1 prior. Policy: auto-approve if "
            "delay>48h AND claim<$500 AND prior<3; else escalate. Which auto-approve? "
            "List the letters only.", gold="A", klass="hard")]
)
print(f"{len(TASKS)} tasks: ", pd.Series([t["klass"] for t in TASKS]).value_counts().to_dict())


12 tasks:  {'simple': 8, 'medium': 3, 'hard': 1}


---
## 2. Baseline — everything on the near-frontier tier (what most teams actually do)

**About to happen.** All 12 tasks on the mid-tier model. We log total cost and accuracy against gold answers.

**Watch for:** `BASELINE cost=` and `accuracy=`. Write the cost on the board (or in a notebook cell comment). Everything later is compared to this.

**Why it matters.** This is the architecture most teams ship: pick the best model that worked in the prototype, use it for everything.


In [4]:
def graded(answer, gold):
    return gold.lower() in answer.lower()

baseline = []
for t in TASKS:
    r = complete(t["q"], model=MODELS.mid, max_tokens=80, label=f"base {t['klass']}")
    ok = graded(r.text, t["gold"])
    baseline.append(dict(klass=t["klass"], model=MODELS.mid, cost=r.usd, ok=ok, ans=r.text[:60]))

bdf = pd.DataFrame(baseline)
B_COST, B_ACC = bdf.cost.sum(), bdf.ok.mean()
print(f"BASELINE  cost={usd(B_COST)}  accuracy={B_ACC:.0%}  ({len(TASKS)} tasks, all near-frontier)")


⚠ No API key — using a rehearsal result.
base simple                                   $0.000848   in=24      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
base simple                                   $0.000848   in=24      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
base simple                                   $0.000848   in=24      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
base simple                                   $0.000848   in=24      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
base simple                                   $0.000848   in=24      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
base simple                                   $0.000848   in=24      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.

---
## 3. The cascade — cheap first, escalate on low confidence

**About to happen.** Each task starts on the floor model. We ask that same cheap model "HIGH / MEDIUM / LOW confidence?" If below 0.9, we escalate to mid; if still shaky, to frontier. The probe is crude — and it is what a production cascade does before paying to escalate.

The confidence *signal* matters more than the routing logic. UCCI (arXiv:2605.18796) showed calibrated uncertainty beat entropy, conformal prediction, and FrugalGPT-style routing at identical accuracy — 31% cost reduction (95% CI 27–35%) on 75,000 real queries. Budget for ~31% on a real workload, not the 85–98% in benchmark papers.

**Watch for:** the path printed per task (`floor` vs `floor>near_frontier`). In most runs, 15–25% of queries leave the floor. That is the RouteLLM result, live.

> **Presenting:** ship the quality gate in the same pull request as the router. A router without an eval harness is an unmonitored quality regression.


In [5]:
def confidence_probe(model, prompt, answer):
    probe = (f"Question: {prompt}\n\nProposed answer: {answer}\n\n"
             "Is this answer fully determined by the question, with no ambiguity? "
             "Reply with only HIGH, MEDIUM or LOW.")
    r = complete(probe, model=model, max_tokens=8, label="probe")
    score = {"HIGH": 0.95, "MEDIUM": 0.6, "LOW": 0.2}.get(r.text.strip().upper()[:6], 0.5)
    return score, r.usd

THRESHOLD = 0.9   # tune on a validation set — do not guess it in production

routed = []
for t in TASKS:
    spend, path = 0.0, []

    r = complete(t["q"], model=MODELS.floor, max_tokens=80, label=f"floor {t['klass']}")
    spend += r.usd
    path.append("floor")
    conf, probe_cost = confidence_probe(MODELS.floor, t["q"], r.text)
    spend += probe_cost
    ans = r.text

    if conf < THRESHOLD:
        r = complete(t["q"], model=MODELS.mid, max_tokens=80, label=f"mid {t['klass']}")
        spend += r.usd
        path.append("near_frontier")
        ans = r.text
        conf, probe_cost = confidence_probe(MODELS.floor, t["q"], ans)
        spend += probe_cost
        if conf < 0.7:
            r = complete(t["q"], model=MODELS.frontier, max_tokens=80, label=f"front {t['klass']}")
            spend += r.usd
            path.append("frontier")
            ans = r.text

    ok = graded(ans, t["gold"])
    routed.append(dict(klass=t["klass"], path=">".join(path), tiers=len(path), cost=spend, ok=ok))
    print(f"{t['klass']:<8} {'>'.join(path):<32} {usd(spend):>11}  {'OK' if ok else 'MISS'}")

rdf = pd.DataFrame(routed)
R_COST, R_ACC = rdf.cost.sum(), rdf.ok.mean()


⚠ No API key — using a rehearsal result.
floor simple                                  $0.000424   in=24      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
probe                                         $0.000470   in=70      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
mid simple                                    $0.000848   in=24      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
probe                                         $0.000470   in=70      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
front simple                                  $0.002120   in=24      out=80     cw=0       cr=0       offline fallback
simple   floor>near_frontier>frontier       $0.004332  MISS
⚠ No API key — using a rehearsal result.
floor simple                                  $0.000424   in=24      out=80     cw=0       cr=0    

---
## 4. The result — and the reframe

**About to happen.** Routing mix (which path each class took), blended cost, raw saving vs baseline.

**Watch for:** raw cost saving, **and** whether accuracy held. A cheaper cascade that misses more is not automatically a win — we prove that in the next cell.

Expect ~31% on a real workload, not the 85–98% you see in papers with a huge cheap/expensive spread.


In [6]:
print("ROUTING MIX")
show(rdf.groupby("path").agg(n=("cost", "size"), cost=("cost", "sum"), acc=("ok", "mean")))

print(f"\n{'':<22}{'cost':>12}{'accuracy':>11}{'cost/task':>12}")
print(f"{'baseline (all mid)':<22}{usd(B_COST):>12}{B_ACC:>11.0%}{usd(B_COST / len(TASKS)):>12}")
print(f"{'3-tier cascade':<22}{usd(R_COST):>12}{R_ACC:>11.0%}{usd(R_COST / len(TASKS)):>12}")
print(f"\nRaw cost saving: {1 - R_COST / B_COST:.0%}")
print("Expect ~31% on a real workload, not the 85-98% you see in benchmark papers.")


ROUTING MIX


,n,cost,acc
path,,,
floor>near_frontier>frontier,12,0.053244,0.083333



                              cost   accuracy   cost/task
baseline (all mid)         $0.0104         8%   $0.000869
3-tier cascade             $0.0532         8%   $0.004437

Raw cost saving: -411%
Expect ~31% on a real workload, not the 85-98% you see in benchmark papers.


\
### Now divide by the success rate

$$E[\text{cost per solved task}] = \frac{C_{\text{attempt}}}{p_{\text{success}}} + L \times K_{\text{cleanup}}$$

`L` is the leak rate of wrong outputs that reach a business process. `K_cleanup` is what a human costs to unwind one. **Cleanup frequently exceeds the API bill.**

**About to happen.** We recompute both routes with `CLEANUP_COST = $12`. Then try 0 and 50 in your head (the next cell does it for you).

**Watch for:** whether the cascade is cheaper *per call* and more expensive *per solved task*. That is the trap. Reliability is a cost lever.

> **Presenting:** set `CLEANUP_COST` out loud with the room. Support vs medical vs finance will pick different numbers, and that is the point.


In [7]:
CLEANUP_COST = 12.00   # USD of human time to unwind one wrong answer. Set this honestly.

def per_solved(total_cost, accuracy, n, cleanup=CLEANUP_COST):
    attempt = total_cost / n
    leaked = 1 - accuracy
    return attempt / max(accuracy, 1e-9) + leaked * cleanup

b = per_solved(B_COST, B_ACC, len(TASKS))
r = per_solved(R_COST, R_ACC, len(TASKS))

print(f"{'':<22}{'cost/CALL':>13}{'cost/SOLVED TASK':>20}")
print(f"{'baseline':<22}{usd(B_COST / len(TASKS)):>13}{usd(b):>20}")
print(f"{'cascade':<22}{usd(R_COST / len(TASKS)):>13}{usd(r):>20}")
print()
if r < b:
    print(f"The cascade wins on BOTH units. Saving per solved task: {1 - r / b:.0%}")
else:
    print("The cascade is cheaper per call and MORE EXPENSIVE per solved task.")
    print("This is the trap. Reliability is a cost lever. Raise the escalation threshold.")
print()
print("Try setting CLEANUP_COST to 0, then to 50. Watch the correct decision flip.")


                          cost/CALL    cost/SOLVED TASK
baseline                  $0.000869              $11.01
cascade                   $0.004437              $11.05

The cascade is cheaper per call and MORE EXPENSIVE per solved task.
This is the trap. Reliability is a cost lever. Raise the escalation threshold.

Try setting CLEANUP_COST to 0, then to 50. Watch the correct decision flip.


---
## 5. Sensitivity — where does the decision actually flip?

**About to happen.** The same two routes, cleanup cost swept from $0 to $100. A column `cascade_wins` tells you when the decision flips.

**Watch for:** the first row where `cascade_wins` changes. That threshold — not the raw saving — is what goes to the decision meeting.

**Why it matters.** "We should route" is not a universal answer. It depends on how expensive a wrong answer is. This table is the artefact.


In [8]:
rows = []
for cu in [0, 1, 5, 12, 25, 50, 100]:
    rows.append(dict(
        cleanup=cu,
        baseline=per_solved(B_COST, B_ACC, len(TASKS), cu),
        cascade=per_solved(R_COST, R_ACC, len(TASKS), cu),
    ))
sdf = pd.DataFrame(rows)
sdf["cascade_wins"] = sdf.cascade < sdf.baseline
show(sdf.style.format({"baseline": "${:,.4f}", "cascade": "${:,.4f}"}))
print()
print("This table, not the raw cost saving, is what you take to the decision meeting.")
print("Ship the quality gate in the same pull request as the router.")


,cleanup,baseline,cascade,cascade_wins
0,0,$0.0104,$0.0532,False
1,1,$0.9271,$0.9699,False
2,5,$4.5938,$4.6366,False
3,12,$11.0104,$11.0532,False
4,25,$22.9271,$22.9699,False
5,50,$45.8438,$45.8866,False
6,100,$91.6771,$91.7199,False



This table, not the raw cost saving, is what you take to the decision meeting.
Ship the quality gate in the same pull request as the router.


In [9]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.0637


,label,model,input,output,cache_write,cache_read,usd,note
0,base simple,claude-sonnet-5,24,80,0,0,0.000848,offline fallback
1,base simple,claude-sonnet-5,24,80,0,0,0.000848,offline fallback
2,base simple,claude-sonnet-5,24,80,0,0,0.000848,offline fallback
3,base simple,claude-sonnet-5,24,80,0,0,0.000848,offline fallback
4,base simple,claude-sonnet-5,24,80,0,0,0.000848,offline fallback
...,...,...,...,...,...,...,...,...
67,floor hard,claude-haiku-4-5,75,80,0,0,0.000475,offline fallback
68,probe,claude-haiku-4-5,121,80,0,0,0.000521,offline fallback
69,mid hard,claude-sonnet-5,75,80,0,0,0.000950,offline fallback
70,probe,claude-haiku-4-5,121,80,0,0,0.000521,offline fallback


---
## Takeaways

- Ship the **quality gate in the same pull request** as the router. A router without an eval harness is an unmonitored quality regression.
- Route on **task class, risk, latency budget and data sensitivity** — never input length alone.
- Calibrate the confidence signal on a validation set. Do not guess the threshold in production.
- Expect **~31% on a real workload**, not the 85–98% in benchmark papers with wide model spreads.
- The number that changes the decision is **cost per successful task**, including cleanup.

**Try on Monday:** sample 100 production queries, label them simple/medium/hard, price them all-on-mid vs 70/20/10. Then ask one operations lead what one wrong answer costs to unwind, and re-rank.
